# Tarea 4: Análisis de Datos y Optimización

## **[Katia Andrea Galindo Ponce]**

**Fecha de entrega:** [VER CANVAS]

**Puntaje total:** 20 puntos

**Instrucciones:**
- Completa los tres problemas en este notebook
- Escribe tu código en las celdas indicadas
- Ejecuta todas las celdas para verificar que tu código funciona
- Guarda tu archivo `.ipynb` en la carpeta `tareas` de tu repositorio privado de GitHub (compartido con el docente)
- Envía el enlace a tu notebook en Canvas

**⚠️ IMPORTANTE:** GitHub registra el historial de cambios de cada archivo. Tu notebook debe ser subido a GitHub **antes del plazo**. **NO** modifiques el archivo después del plazo — los cambios tardíos serán detectados y pueden resultar en penalidad.

**Integridad académica:** Esta es una tarea individual. Puedes consultar los materiales del curso, documentación de Python, herramientas de IA y discutir conceptos con compañeros, pero todo el código debe ser tuyo.

---

In [1]:
# Importaciones estándar - ejecuta esta celda primero
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize, curve_fit
from io import StringIO

---
## Problema 1: Análisis de Calidad de Agua en Ríos Andino-Amazónicos (7 puntos)

Estás analizando datos de calidad de agua de estaciones de monitoreo en tres ríos de la cuenca amazónica peruana, provenientes de la red de monitoreo de la **Autoridad Nacional del Agua (ANA)**. El conjunto de datos contiene mediciones de temperatura, oxígeno disuelto (OD), pH y conductividad eléctrica recolectadas durante varios meses.

### Tus Tareas:

**Parte A (2 puntos):** Carga y explora los datos
1. Carga los datos del string CSV provisto abajo en un DataFrame de pandas
2. Muestra información básica del conjunto de datos (forma, tipos de datos, primeras filas)
3. Verifica valores faltantes e indica cuántos hay en cada columna
4. Convierte la columna `fecha` a formato datetime usando `pd.to_datetime()`

**Parte B (3 puntos):** Análisis de datos con agrupación
1. Calcula la media, desviación estándar, mínimo y máximo de oxígeno disuelto (`od_mg_l`) agrupado por `estacion_id`
2. Determina qué estación tiene la media más baja de oxígeno disuelto
3. Crea una nueva columna llamada `estado_od` que clasifique cada medición como:
   - "Crítico" si OD < 4 mg/L
   - "Bajo" si OD está entre 4 y 6 mg/L
   - "Adecuado" si OD está entre 6 y 8 mg/L
   - "Bueno" si OD >= 8 mg/L
4. Cuenta cuántas mediciones caen en cada categoría de `estado_od` por estación

**Parte C (2 puntos):** Filtrado y resumen
1. Filtra los datos para incluir solo mediciones donde temperatura > 20°C Y pH entre 6.5 y 8.5
2. Para este subconjunto filtrado, calcula la conductividad media por mes (pista: extrae el mes de la fecha)
3. Identifica qué combinación estación-mes tuvo el mayor número de lecturas con OD "Crítico" o "Bajo"

In [50]:
# Dataset de calidad de agua - ríos andino-amazónicos del Perú
calidad_agua_csv = (
    "estacion_id,fecha,temp_c,od_mg_l,ph,conductividad_us\n"
    "RIO_UCAYALI,2024-05-15,24.3,7.2,7.1,145\n"
    "RIO_UCAYALI,2024-05-22,25.1,6.8,7.0,152\n"
    "RIO_UCAYALI,2024-06-05,24.8,6.5,6.9,158\n"
    "RIO_UCAYALI,2024-06-19,25.5,5.8,6.8,165\n"
    "RIO_UCAYALI,2024-07-03,24.2,5.2,7.0,172\n"
    "RIO_UCAYALI,2024-07-17,23.8,4.8,7.1,168\n"
    "RIO_UCAYALI,2024-08-01,24.5,4.2,7.2,175\n"
    "RIO_UCAYALI,2024-08-15,25.2,5.0,7.0,169\n"
    "RIO_TAMBOPATA,2024-05-15,21.8,8.5,7.4,98\n"
    "RIO_TAMBOPATA,2024-05-22,22.5,8.1,7.5,105\n"
    "RIO_TAMBOPATA,2024-06-05,22.9,7.8,7.3,112\n"
    "RIO_TAMBOPATA,2024-06-19,23.4,7.2,7.2,118\n"
    "RIO_TAMBOPATA,2024-07-03,22.1,6.8,7.1,125\n"
    "RIO_TAMBOPATA,2024-07-17,21.8,6.5,7.0,121\n"
    "RIO_TAMBOPATA,2024-08-01,22.5,6.9,7.1,128\n"
    "RIO_TAMBOPATA,2024-08-15,23.1,7.2,7.2,115\n"
    "RIO_MANTARO,2024-05-15,13.1,9.5,6.5,312\n"
    "RIO_MANTARO,2024-05-22,14.2,8.8,6.4,325\n"
    "RIO_MANTARO,2024-06-05,12.8,8.2,6.3,338\n"
    "RIO_MANTARO,2024-06-19,13.5,7.5,6.2,352\n"
    "RIO_MANTARO,2024-07-03,11.9,6.8,6.0,368\n"
    "RIO_MANTARO,2024-07-17,10.8,5.9,5.9,378\n"
    "RIO_MANTARO,2024-08-01,11.5,5.2,6.1,385\n"
    "RIO_MANTARO,2024-08-15,12.3,4.8,6.2,372\n"
)

# Parte A: Carga y explora los datos
from io import StringIO
df = pd.read_csv(StringIO(calidad_agua_csv))

#2 Información
print("----DataFrame (Filas, Columnas)----")
print(df.shape)

print("----Tipos de datos iniciales----")
print(df.dtypes)

print("                  ----Primeras 5 filas----")
print(df.head())

# 3. Valores faltantes por columna
print("----Valores faltantes por columna----")
print(df.isnull().sum())

# 4. Convertir la columna 'fecha' a formato datetime
df['fecha'] = pd.to_datetime(df['fecha'])


----DataFrame (Filas, Columnas)----
(24, 6)
----Tipos de datos iniciales----
estacion_id          object
fecha                object
temp_c              float64
od_mg_l             float64
ph                  float64
conductividad_us      int64
dtype: object
                  ----Primeras 5 filas----
   estacion_id       fecha  temp_c  od_mg_l   ph  conductividad_us
0  RIO_UCAYALI  2024-05-15    24.3      7.2  7.1               145
1  RIO_UCAYALI  2024-05-22    25.1      6.8  7.0               152
2  RIO_UCAYALI  2024-06-05    24.8      6.5  6.9               158
3  RIO_UCAYALI  2024-06-19    25.5      5.8  6.8               165
4  RIO_UCAYALI  2024-07-03    24.2      5.2  7.0               172
----Valores faltantes por columna----
estacion_id         0
fecha               0
temp_c              0
od_mg_l             0
ph                  0
conductividad_us    0
dtype: int64


In [51]:
# Parte B: Análisis de datos con agrupación

stats_od = df.groupby('estacion_id')['od_mg_l'].agg(
    Media='mean',
    Desviacion_Estandar='std',
    Minimo='min',
    Maximo='max'
)

# 1. Oxígeno disuelto por estación

print("---- Oxígeno Disuelto por Estación----")
print(stats_od)

# 2. Estación con la media más baja de oxígeno disuelto

estacion_min_media = stats_od['Media'].idxmin()
valor_min_media = stats_od['Media'].min()

print(f"\n ---- Estación con menor media de OD ----")
print(f"Estación: {estacion_min_media} ({valor_min_media:.2f} mg/L)")

# 3. Clasificación de cada medición

condiciones = [
    df['od_mg_l'] < 4,
    (df['od_mg_l'] >= 4) & (df['od_mg_l'] < 6),
    (df['od_mg_l'] >= 6) & (df['od_mg_l'] < 8),
    df['od_mg_l'] >= 8 ]

etiquetas = ['Crítico', 'Bajo', 'Adecuado', 'Bueno']

df['estado_od'] = np.select(condiciones, etiquetas, default='Sin clasificar')

print("Clasificación de cada medición'estado_od' ")
print(df[['estacion_id', 'od_mg_l', 'estado_od']])

# 4. Conteo de mediciones por categoría de estado_od y estación
conteo_estado = pd.crosstab(df['estacion_id'], df['estado_od'])

print("\n ---- Conteo de mediciones por categoría de 'estado_od'----")
print(conteo_estado)

---- Oxígeno Disuelto por Estación----
                Media  Desviacion_Estandar  Minimo  Maximo
estacion_id                                               
RIO_MANTARO    7.0875             1.709166     4.8     9.5
RIO_TAMBOPATA  7.3750             0.692305     6.5     8.5
RIO_UCAYALI    5.6875             1.062931     4.2     7.2

 ---- Estación con menor media de OD ----
Estación: RIO_UCAYALI (5.69 mg/L)
Clasificación de cada medición'estado_od' 
      estacion_id  od_mg_l estado_od
0     RIO_UCAYALI      7.2  Adecuado
1     RIO_UCAYALI      6.8  Adecuado
2     RIO_UCAYALI      6.5  Adecuado
3     RIO_UCAYALI      5.8      Bajo
4     RIO_UCAYALI      5.2      Bajo
5     RIO_UCAYALI      4.8      Bajo
6     RIO_UCAYALI      4.2      Bajo
7     RIO_UCAYALI      5.0      Bajo
8   RIO_TAMBOPATA      8.5     Bueno
9   RIO_TAMBOPATA      8.1     Bueno
10  RIO_TAMBOPATA      7.8  Adecuado
11  RIO_TAMBOPATA      7.2  Adecuado
12  RIO_TAMBOPATA      6.8  Adecuado
13  RIO_TAMBOPATA      6.5  

In [52]:
# Parte C: Filtrado y resumen

# 1. Filtro de los datos para incluir mediciones
df_filtrado = df[(df['temp_c'] > 20) & (df['ph'] >= 6.5) & (df['ph'] <= 8.5)].copy()

# Filtrado de conductividad media por mes
df_filtrado['mes'] = df_filtrado['fecha'].dt.month
conductividad_media_mes = df_filtrado.groupby('mes')['conductividad_us'].mean()

print("----Conductividad media por mes ----")
print(conductividad_media_mes)

# 3. Combinación estación-mes con mayor número de lecturas con OD "Crítico" o "Bajo"

df['mes'] = df['fecha'].dt.month
df_critico_bajo = df[df['estado_od'].isin(['Crítico', 'Bajo'])]

# Agrupar por estación y mes para obtener los conteos
conteo_estacion_mes = df_critico_bajo.groupby(['estacion_id', 'mes']).size()

print("\n---- Conteo de lecturas 'Crítico' / 'Bajo' por Estación y Mes ----")
print(conteo_estacion_mes)

# Identificar la combinación con el valor máximo de lecturas
max_lecturas = conteo_estacion_mes.max()
combinaciones_max = conteo_estacion_mes[conteo_estacion_mes == max_lecturas]

print(f"\nMayor número de lecturas registradas: {max_lecturas}")
print("Combinación estación-mes correspondientes:")
print(combinaciones_max)

----Conductividad media por mes ----
mes
5    125.00
6    138.25
7    146.50
8    146.75
Name: conductividad_us, dtype: float64

---- Conteo de lecturas 'Crítico' / 'Bajo' por Estación y Mes ----
estacion_id  mes
RIO_MANTARO  7      1
             8      2
RIO_UCAYALI  6      1
             7      2
             8      2
dtype: int64

Mayor número de lecturas registradas: 2
Combinación estación-mes correspondientes:
estacion_id  mes
RIO_MANTARO  8      2
RIO_UCAYALI  7      2
             8      2
dtype: int64


---
## Problema 2: Comparación Estadística de Parcelas Forestales en la Amazonía (6 puntos)

Investigadores del **INIA (Instituto Nacional de Innovación Agraria) - Estación Experimental Pucallpa** midieron la biomasa arbórea (kg) en parcelas pareadas — unas sometidas a un tratamiento de aprovechamiento forestal de impacto reducido (AFIR) y otras dejadas como control. Se quiere determinar si el tratamiento afectó significativamente la biomasa individual de los árboles y si existe relación entre el diámetro y la biomasa.

### Tus Tareas:

**Parte A (2 puntos):** Comparación de grupos de tratamiento
1. Calcula estadísticas descriptivas (media, desviación estándar, mediana) de biomasa para cada grupo
2. Realiza una prueba t de dos muestras independientes para determinar si hay diferencia significativa en la biomasa media entre parcelas control y AFIR (α = 0.05)
3. Plantea tu hipótesis nula y alternativa, reporta el estadístico t y el p-valor, y escribe una conclusión

**Parte B (2 puntos):** Análisis de correlación
1. Calcula el coeficiente de correlación de Pearson entre el DAP y la biomasa para todo el conjunto de datos
2. Evalúa si esta correlación es estadísticamente significativa (α = 0.05)
3. Interpreta la fuerza y dirección de la correlación

**Parte C (2 puntos):** Ajuste de distribución
1. Ajusta una distribución normal a los datos de biomasa de las parcelas control
2. Reporta los parámetros ajustados (μ y σ)
3. Calcula la probabilidad de que un árbol seleccionado aleatoriamente de las parcelas control tenga biomasa > 150 kg
4. ¿Qué valor de biomasa representa el percentil 90 para los árboles de parcelas control?

In [26]:
# Datos de parcelas forestales - INIA Pucallpa
np.random.seed(458)  # Para reproducibilidad

# Parcelas control: bosque sin intervención
n_control = 35
dap_control = np.random.uniform(15, 50, n_control)  # DAP en cm
biomasa_control = 0.1 * dap_control**2.2 + np.random.normal(0, 15, n_control)
biomasa_control = np.maximum(biomasa_control, 10)  # Asegurar valores positivos

# Parcelas AFIR: los árboles remanentes disponen de más recursos
n_afir = 30
dap_afir = np.random.uniform(18, 55, n_afir)  # DAP en cm
biomasa_afir = 0.12 * dap_afir**2.2 + np.random.normal(5, 18, n_afir)
biomasa_afir = np.maximum(biomasa_afir, 10)

# Crear DataFrame
bosque_df = pd.DataFrame({
    'dap_cm': np.concatenate([dap_control, dap_afir]),
    'biomasa_kg': np.concatenate([biomasa_control, biomasa_afir]),
    'tratamiento': ['Control']*n_control + ['AFIR']*n_afir
})

print(bosque_df.head())
print(f"\nEspecies representativas: Caoba (Swietenia macrophylla), Cedro (Cedrela odorata), Tornillo (Cedrelinga cateniformis)")

      dap_cm  biomasa_kg tratamiento
0  43.208835  395.956074     Control
1  49.475851  521.291644     Control
2  19.647405   54.799539     Control
3  23.651882  119.264060     Control
4  40.431164  335.576313     Control

Especies representativas: Caoba (Swietenia macrophylla), Cedro (Cedrela odorata), Tornillo (Cedrelinga cateniformis)


In [53]:
# Parte A: Comparación de grupos de tratamiento

# Estadísticas descriptivas de biomasa por grupo de tratamiento
stats_biomasa = bosque_df.groupby('tratamiento')['biomasa_kg'].agg(
    Media='mean',
    Desviacion_Estandar='std',
    Mediana='median'
)

print("---- Descripción de Biomasa (kg) ----")
print(stats_biomasa)

# Separación de grupos
biomasa_control = bosque_df[bosque_df['tratamiento'] == 'Control']['biomasa_kg']
biomasa_afir = bosque_df[bosque_df['tratamiento'] == 'AFIR']['biomasa_kg']

# Prueba t de muestras independientes
t_stat, p_valor = stats.ttest_ind(biomasa_control, biomasa_afir)

print("\n---- Resultados de la Prueba t ----")
print(f"Estadístico t: {t_stat:.4f}")
print(f"P-valor: {p_valor:.4e}")

# Evaluación con nivel de significancia alpha = 0.05
alpha = 0.05
if p_valor < alpha:
    print(f"\nConclusión: Como el p-valor ({p_valor:.4e}) es menor que alpha ({alpha}), se RECHAZA la hipótesis nula (H0). Existe una diferencia estadísticamente significativa en la biomasa media entre las parcelas Control y AFIR.")
else:
    print(f"\nConclusión: Como el p-valor ({p_valor:.4e}) es mayor o igual que alpha ({alpha}), NO se rechaza la hipótesis nula (H0). No existe evidencia suficiente para afirmar una diferencia significativa en la biomasa media entre parcelas.")

---- Descripción de Biomasa (kg) ----
                  Media  Desviacion_Estandar     Mediana
tratamiento                                             
AFIR         325.418953           213.516434  239.121336
Control      269.642640           158.210017  232.870089

---- Resultados de la Prueba t ----
Estadístico t: -1.2070
P-valor: 2.3194e-01

Conclusión: Como el p-valor (2.3194e-01) es mayor o igual que alpha (0.05), NO se rechaza la hipótesis nula (H0). No existe evidencia suficiente para afirmar una diferencia significativa en la biomasa media entre parcelas.


In [55]:
# Parte B: Análisis de correlación
# Coeficiente de correlación de Pearson y p-valor para todo el conjunto de datos
r_pearson, p_valor = stats.pearsonr(bosque_df['dap_cm'], bosque_df['biomasa_kg'])

print("---- Análisis de Correlación de Pearson ----")
print(f"Coeficiente de correlación (r): {r_pearson:.4f}")
print(f"p-valor: {p_valor:.4e}")

# Evaluación de significancia estadística (alpha = 0.05)
alpha = 0.05
if p_valor < alpha:
    print(f"\nResultado: La correlación es estadisticamente significativa (p-valor < {alpha}).")
else:
    print(f"\nResultado: La correlación NO es estadísticamente significativa (p-valor >= {alpha}).")


---- Análisis de Correlación de Pearson ----
Coeficiente de correlación (r): 0.9565
p-valor: 2.0976e-35

Resultado: La correlación es estadisticamente significativa (p-valor < 0.05).


In [56]:
# Parte C: Ajuste de distribución

# 1. Filtrar los datos de biomasa para las parcelas control
biomasa_control = bosque_df[bosque_df['tratamiento'] == 'Control']['biomasa_kg']

# 2. Parámetros ajustados
mu, sigma = stats.norm.fit(biomasa_control)

print("----Parámetros de la Distribución Normal Ajustada ----")
print(f"Media (μ): {mu:.2f} kg")
print(f"Desviación estándar (σ): {sigma:.2f} kg")

# 3. Probabilidad de que un árbol tenga biomasa > 150 kg: P(X > 150)
prob_mayor_150 = stats.norm.sf(150, loc=mu, scale=sigma)

print("\n---- Probabilidad de Biomasa > 150 kg ----")
print(f"P(X > 150 kg) = {prob_mayor_150:.4f} ({prob_mayor_150 * 100:.2f}%)")

# 4. Percentil 90
percentil_90 = stats.norm.ppf(0.90, loc=mu, scale=sigma)

print("\n----  Percentil 90 ----")
print(f"El percentil 90 (modelo normal) es: {percentil_90:.2f} kg")

----Parámetros de la Distribución Normal Ajustada ----
Media (μ): 269.64 kg
Desviación estándar (σ): 155.93 kg

---- Probabilidad de Biomasa > 150 kg ----
P(X > 150 kg) = 0.7785 (77.85%)

----  Percentil 90 ----
El percentil 90 (modelo normal) es: 469.48 kg


---
## Problema 3: Ajuste de Curva de Respuesta a la Luz (7 puntos)

La fotosíntesis depende de la intensidad de luz siguiendo una curva de saturación. La **hipérbola rectangular** se usa comúnmente para modelar esta relación:

$$A = \frac{A_{max} \cdot I}{K + I} - R_d$$

Donde:
- $A$ = tasa de fotosíntesis neta (μmol CO₂ m⁻² s⁻¹)
- $A_{max}$ = tasa máxima de fotosíntesis a saturación de luz
- $I$ = intensidad de luz (μmol fotones m⁻² s⁻¹, PAR)
- $K$ = constante de media saturación (nivel de luz en el que A = A_max/2 - R_d)
- $R_d$ = tasa de respiración en oscuridad (CO₂ liberado cuando I = 0)

Los datos provienen de mediciones de *Cecropia sp.* ("cetico"), una especie pionera característica de la Amazonía peruana, muy importante en la regeneración de bosques perturbados.

### Tus Tareas:

**Parte A (2 puntos):** Define el modelo y la función de costo
1. Escribe una función `respuesta_luz(I, Amax, K, Rd)` que implemente la ecuación anterior
2. Escribe una función de costo `respuesta_luz_mse(params, I_datos, A_datos)` que calcule el error cuadrático medio entre las tasas de fotosíntesis observadas y predichas
3. Prueba tu función `respuesta_luz` calculando A para I = 500 con Amax=25, K=200, Rd=2

**Parte B (3 puntos):** Ajusta el modelo usando optimización
1. Usa `scipy.optimize.minimize` para encontrar los parámetros óptimos (Amax, K, Rd) que minimicen el MSE
2. Usa valores iniciales: Amax=20, K=150, Rd=1
3. Reporta los parámetros ajustados y el MSE final
4. Ajusta también el modelo usando `scipy.optimize.curve_fit` y compara los resultados

**Parte C (2 puntos):** Evalúa e interpreta el modelo
1. Calcula los valores de fotosíntesis predichos usando tus parámetros ajustados
2. Calcula R² (coeficiente de determinación) para evaluar el ajuste del modelo:
   $$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$
3. Calcula el **punto de compensación lumínico** (el nivel de luz donde A = 0, es decir, la fotosíntesis iguala a la respiración). Pista: despeja I cuando A = 0
4. ¿Cuál es la tasa de fotosíntesis a saturación lumínica (Amax - Rd)?

In [36]:
# Datos de curva de respuesta a la luz - Cecropia sp. (cetico)
# Mediciones en parcela de investigación, Madre de Dios

# PAR (radiación fotosintéticamente activa) en μmol fotones m⁻² s⁻¹
par_datos = np.array([0, 25, 50, 75, 100, 150, 200, 300, 400, 600, 800, 1000, 1200, 1500, 1800])

# Tasa de fotosíntesis neta en μmol CO₂ m⁻² s⁻¹
foto_datos = np.array([-1.8, 1.2, 4.5, 7.1, 9.2, 12.5, 14.8, 17.5, 19.2, 21.1, 22.0, 22.5, 22.8, 23.0, 23.1])

print(f"Rango PAR: {par_datos.min()} a {par_datos.max()} μmol fotones m⁻² s⁻¹")
print(f"Rango fotosíntesis: {foto_datos.min()} a {foto_datos.max()} μmol CO₂ m⁻² s⁻¹")
print("Especie: Cecropia sp. (cetico) - pionera amazónica")

Rango PAR: 0 a 1800 μmol fotones m⁻² s⁻¹
Rango fotosíntesis: -1.8 a 23.1 μmol CO₂ m⁻² s⁻¹
Especie: Cecropia sp. (cetico) - pionera amazónica


In [39]:
# Parte A: Define el modelo y la función de costo

# 1. Modelo de respuesta a la luz (Hipérbola rectangular)
def respuesta_luz(I, Amax, K, Rd):
    return (Amax * I) / (K + I) - Rd

# 2.
def respuesta_luz_mse(params, I_datos, A_datos):
    Amax, K, Rd = params
    A_pred = respuesta_luz(I_datos, Amax, K, Rd)
    return np.mean((A_datos - A_pred) ** 2)

# 3.
I_test = 500
Amax_test = 25
K_test = 200
Rd_test = 2

A_resultado = respuesta_luz(I_test, Amax_test, K_test, Rd_test)

print(f"Para I = {I_test} µmol fotones m⁻² s⁻¹:")
print(f"Fotosíntesis neta calculada (A) = {A_resultado:.4f} µmol CO₂ m⁻² s⁻¹")

Para I = 500 µmol fotones m⁻² s⁻¹:
Fotosíntesis neta calculada (A) = 15.8571 µmol CO₂ m⁻² s⁻¹


In [58]:
# Parte B: Ajusta el modelo usando optimización

# Datos de entrada
par_datos = np.array([0, 25, 50, 75, 100, 150, 200, 300, 400, 600, 800, 1000, 1200, 1500, 1800])
foto_datos = np.array([-1.8, 1.2, 4.5, 7.1, 9.2, 12.5, 14.8, 17.5, 19.2, 21.1, 22.0, 22.5, 22.8, 23.0, 23.1])

# Valuación inicial: [Amax, K, Rd]
p0 = [20, 150, 1]

res_minimize = minimize(respuesta_luz_mse, x0=p0, args=(par_datos, foto_datos))
Amax_min, K_min, Rd_min = res_minimize.x
mse_min = res_minimize.fun

print("\n---- Resultados con scipy.optimize.minimize----")
print(f"Amax optimizado : {Amax_min:.4f} µmol CO₂ m⁻² s⁻¹")
print(f"K optimizado    : {K_min:.4f} µmol fotones m⁻² s⁻¹")
print(f"Rd optimizado   : {Rd_min:.4f} µmol CO₂ m⁻² s⁻¹")
print(f"MSE final       : {mse_min:.6f}")

# 4.
popt_cf, pcov_cf = curve_fit(respuesta_luz, par_datos, foto_datos, p0=p0)
Amax_cf, K_cf, Rd_cf = popt_cf
mse_cf = respuesta_luz_mse(popt_cf, par_datos, foto_datos)

print("\n---- Resultados con scipy.optimize.curve_fit ----")
print(f"Amax optimizado : {Amax_cf:.4f} µmol CO₂ m⁻² s⁻¹")
print(f"K optimizado    : {K_cf:.4f} µmol fotones m⁻² s⁻¹")
print(f"Rd optimizado   : {Rd_cf:.4f} µmol CO₂ m⁻² s⁻¹")
print(f"MSE final       : {mse_cf:.6f}")


---- Resultados con scipy.optimize.minimize----
Amax optimizado : 28.4461 µmol CO₂ m⁻² s⁻¹
K optimizado    : 134.7545 µmol fotones m⁻² s⁻¹
Rd optimizado   : 2.6272 µmol CO₂ m⁻² s⁻¹
MSE final       : 0.237337

---- Resultados con scipy.optimize.curve_fit ----
Amax optimizado : 28.4460 µmol CO₂ m⁻² s⁻¹
K optimizado    : 134.7545 µmol fotones m⁻² s⁻¹
Rd optimizado   : 2.6272 µmol CO₂ m⁻² s⁻¹
MSE final       : 0.237337


In [49]:
# Parte C: Evalúa e interpreta el modelo

Amax, K, Rd = popt_cf

# 1. Valores predichos de fotosíntesis
foto_pred = respuesta_luz(par_datos, Amax, K, Rd)

# 2. Cálculo del coeficiente de determinación
ss_res = np.sum((foto_datos - foto_pred) ** 2)
ss_tot = np.sum((foto_datos - np.mean(foto_datos)) ** 2)
r2 = 1 - (ss_res / ss_tot)

# 3. Punto de compensación lumínico
I_compensacion = (Rd * K) / (Amax - Rd)

# 4. Tasa de fotosíntesis a saturación lumínica (Amax - Rd)
A_saturacion = Amax - Rd

print("----Valores predichos de fotosíntesis----")
print(np.round(foto_pred, 4))

print("\n----Evaluación del Modelo----")
print(f"R² (Coeficiente de determinación) : {r2:.6f}")
print(f"Punto de compensación lumínico (I): {I_compensacion:.4f} µmol fotones m⁻² s⁻¹")
print(f"Fotosíntesis a saturación lumínica : {A_saturacion:.4f} µmol CO₂ m⁻² s⁻¹")


----Valores predichos de fotosíntesis----
[-2.6272  1.8244  5.0712  7.544   9.4902 12.3573 14.368  17.0019 18.6507
 20.6019 21.7181 22.4408 22.947  23.474  23.8376]

----Evaluación del Modelo----
R² (Coeficiente de determinación) : 0.996575
Punto de compensación lumínico (I): 13.7117 µmol fotones m⁻² s⁻¹
Fotosíntesis a saturación lumínica : 25.8189 µmol CO₂ m⁻² s⁻¹


---
## Lista de verificación para entrega

Antes de entregar, verifica que:

- [ ] Todas las celdas de código se ejecutan sin errores
- [ ] Los tres problemas están completos
- [ ] Los resultados son visibles en todas las celdas
- [ ] Tu nombre está incluido al inicio
- [ ] El archivo está guardado en la carpeta `tareas` de tu repositorio privado de GitHub
- [ ] El archivo está subido a GitHub **antes del plazo**
- [ ] El enlace a tu notebook está enviado en Canvas